# Notebook 14: raw qHTS and aux beside the `chemprop_medium` embedding

**If Colab disconnects:** reconnect, choose Runtime, then Run all, and approve the Drive prompt again.
Every fold is saved to `MyDrive/cyp_results` the moment it finishes, so the run picks up at the next
unfinished fold. A finished isoform is skipped completely. Nothing needs editing between attempts.

**What it does.** TabPFN regressor, scaffold 5-fold out-of-fold predictions on the development rows,
a holdout prediction, and a blind-test prediction, for each of the four isoforms. Output files
`nb14_{iso}.npz` carry the keys `oof, dev, hold, ho, test`, the same layout as every earlier arm, so
they drop straight into the blend. The features are the embedding reduced to 200 principal components,
with the five qHTS probability columns and the four primary-screen columns kept raw beside them.
`tabfull.py` found this layout took CYP1A2 from 0.5610 to 0.5816 against putting everything through the
PCA. It was only measured on CYP1A2, and this run measures it on all four.

**Time.** About 1 to 1.5 hours on a Colab GPU, estimated from nb12's per-isoform times. Not measured for this layout.

**Reading the result.** The last line prints the macro dev Spearman against 0.6273, the macro of the production
`tabqhts` arm (0.5610, 0.7189, 0.4219, 0.8074). A gain here is only worth anything if the arm also adds to the blend.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to a GPU before going on')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/rawq_medium_02ee00be6c.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
from google.colab import drive
drive.mount('/content/drive')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
assert os.path.ismount('/content/drive'), 'Drive is not mounted; stop here, the run could not resume without it'
print('writing to', OUTDIR, '| files already there:', sorted(f for f in os.listdir(OUTDIR) if f.startswith(('nb14', 'nb15', 'part_', 'results_nb1')))[:12])

In [ ]:
!pip -q install tabpfn==2.2.1

In [ ]:
import json, os, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from tabpfn import TabPFNRegressor

TAG = 'nb14'
PRE_PCA = False          # True when the embedding in the payload is already 200 PCA components
CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ISO = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
REF = {'CYP1A2': 0.5610, 'CYP2C9': 0.7189, 'CYP2D6': 0.4219, 'CYP3A4': 0.8074}   # tabqhts dev Spearman
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
meta = pd.read_csv(CACHE / 'meta_train.csv.gz'); hold = np.load(CACHE / 'holdout_mask.npy')
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
EMB = np.load(CACHE / 'emb_train.npy').astype(np.float32); EMB_TE = np.load(CACHE / 'emb_test.npy').astype(np.float32)
RAW = np.hstack([np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'aux_train.npy')]).astype(np.float32)
RAW_TE = np.hstack([np.load(CACHE / 'qhts_te.npy'), np.load(CACHE / 'aux_test.npy')]).astype(np.float32)
print('device', DEVICE, '| emb', EMB.shape, '| raw qhts+aux', RAW.shape, '| test', EMB_TE.shape)
assert len(EMB) == len(meta) == len(RAW) and len(EMB_TE) == len(RAW_TE)

def feats(tr_idx, ev_X, ev_RAW):
    """Fit the transform on training rows only; return (train features, eval features)."""
    a, b = EMB[tr_idx], ev_X
    if not PRE_PCA:
        n = min(200, a.shape[0], a.shape[1])
        imp = SimpleImputer(strategy='median').fit(a); sc = StandardScaler().fit(imp.transform(a))
        pca = PCA(n_components=n, random_state=42).fit(sc.transform(imp.transform(a)))
        f = lambda m: pca.transform(sc.transform(imp.transform(m))).astype(np.float32)
        a, b = f(a), f(b)
    ri = SimpleImputer(strategy='median').fit(RAW[tr_idx])
    return np.hstack([a, ri.transform(RAW[tr_idx])]).astype(np.float32), np.hstack([b, ri.transform(ev_RAW)]).astype(np.float32)

KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)
def fit_predict(tr_idx, ev_X, ev_RAW, y):
    Xt, Xe = feats(tr_idx, ev_X, ev_RAW)
    return TabPFNRegressor(**KW).fit(Xt, y[tr_idx]).predict(Xe)

# smoke test: 90 rows, one fit, every shape printed, fails loudly before the long run
y0 = meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float); ok = np.flatnonzero(np.isfinite(y0))[:90]
Xt, Xe = feats(ok[:70], EMB[ok[70:]], RAW[ok[70:]])
p = TabPFNRegressor(**KW).fit(Xt, y0[ok[:70]]).predict(Xe)
print('smoke test ok: train', Xt.shape, 'eval', Xe.shape, 'pred', p.shape, 'finite', bool(np.isfinite(p).all()))

def save_atomic(path, **kw):
    tmp = Path(str(path) + '.tmp.npz'); np.savez_compressed(tmp, **kw); os.replace(tmp, path)

RESULTS = []
t_all = time.time()
for iso in ISO:
    final = OUT / f'{TAG}_{iso}.npz'
    y = meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
    obs = np.isfinite(y); dev = np.flatnonzero(obs & ~hold); ho = np.flatnonzero(obs & hold); allr = np.flatnonzero(obs)
    if final.exists():
        z = np.load(final); oof, hp = z['oof'], z['hold']; print(f'{iso}: finished earlier, skipping', flush=True)
    else:
        part = OUT / f'part_{TAG}_{iso}.npz'
        oof = np.full(len(dev), np.nan); hp = np.full(len(ho), np.nan); tp = np.full(len(EMB_TE), np.nan); done = []
        if part.exists():
            z = np.load(part, allow_pickle=True); oof, hp, tp = z['oof'], z['hp'], z['tp']; done = list(z['done'])
            print(f'{iso}: resuming, finished stages {done}', flush=True)
        def ckpt(): save_atomic(part, oof=oof, hp=hp, tp=tp, done=np.array(done))
        t0 = time.time()
        for k, (tr, va) in enumerate(GroupKFold(5).split(dev, dev, G[dev])):
            if f'f{k}' in done: continue
            oof[va] = fit_predict(dev[tr], EMB[dev[va]], RAW[dev[va]], y)
            done.append(f'f{k}'); ckpt(); print(f'  {iso} fold {k} ({time.time()-t0:.0f}s)', flush=True)
        if 'hold' not in done:
            hp = fit_predict(dev, EMB[ho], RAW[ho], y); done.append('hold'); ckpt(); print(f'  {iso} holdout model ({time.time()-t0:.0f}s)', flush=True)
        if 'test' not in done:
            tp = fit_predict(allr, EMB_TE, RAW_TE, y); done.append('test'); ckpt(); print(f'  {iso} test model ({time.time()-t0:.0f}s)', flush=True)
        assert np.isfinite(oof).all() and np.isfinite(hp).all() and np.isfinite(tp).all()
        save_atomic(final, oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
        part.unlink(missing_ok=True)
    d = float(spearmanr(y[dev], oof).statistic); h = float(spearmanr(y[ho], hp).statistic)
    RESULTS.append(dict(tag=TAG, iso=iso, dev_spearman=d, holdout_spearman=h, ref_tabqhts=REF[iso]))
    json.dump(RESULTS, open(OUT / f'results_{TAG}.json', 'w'), indent=2)
    print(f'{iso}: dev {d:.4f} (tabqhts {REF[iso]:.4f}) | holdout {h:.4f}', flush=True)
r = pd.DataFrame(RESULTS)
print('\nmacro dev', round(r.dev_spearman.mean(), 4), '| tabqhts macro', round(np.mean(list(REF.values())), 4),
      '| total time', round((time.time() - t_all) / 60, 1), 'min')

In [ ]:
import shutil
shutil.make_archive(f'/content/{TAG}_results', 'zip', OUT)
try:
    from google.colab import files
    files.download(f'/content/{TAG}_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- the results are already on Drive in', OUT)